განზომილებები დეშბორდის ფილტრებისა და კავშირებისთვის.

ვარსკვლავური სქემა: ფაქტები CalcMembers-ის გრანულარობით
(სტრიქონი = ერთი მთამსვლელი ერთ ექსპედიციაში) და ოთხი განზომილება,
რომლებიც მათ გასაღებით უკავშირდება.

განზომილება	გასაღები	რომელი dataset უკავშირდება
CalcDimPeak	PeakId	ds_members, ds_decades, ds_peak_risk, ds_peak_summary, ds_georgians
CalcDimDecade	Decade	ds_members, ds_decades, ds_peak_risk, ds_peak_summary, ds_georgians
CalcDimNation	Nation	ds_members, ds_decades

CalcDimYear რჩება როგორც CalcDimDecade-ის წყარო. თავად დეშბორდში კი ათწლეულის განზომილება გამოიყენება — ფილტრი Decade/Era-ზეა და არა ნედლ წელზე.

კავშირები დეშბორდში იწერება Data → Relationships-ში, ყველა many-to-one მიმართულებით — ფაქტიდან განზომილებისკენ.

ეს არის მრავალფაქტიანი ვარსკვლავი: ერთ განზომილებას რამდენიმე ფაქტი უკავშირდება. ფაქტები ერთმანეთს პირდაპირ არ ებმის, რათა გრაფში ციკლი არ გაჩნდეს და join-ის გზა ორაზროვანი არ გახდეს.

დაუკავშირებელი რჩება სამი dataset:

ds_agencies — გრანულარობა სააგენტოა და არა ასვლა.
ds_leaderboard — გრანულარობა ადამიანია.
ds_14x8000 — Nationality ვიკიდან მოდის და არქივის Citizenship-ის დომენში არ არის (Nepal vs Nepalese).

ცრუ კავშირი უარესია, ვიდრე არცერთი.

In [0]:
%sql
-- 1. CalcDimPeak — მწვერვალები (16)
CREATE OR REPLACE TABLE getdata.calculated.CalcDimPeak AS
SELECT
Pk.PeakId,
Pk.PeakName,
Pk.HeightMetres,
Pk.HeightBand,
Pk.MountainRange,
Pk.Region,
Pk.ClimbingStatus,
Pk.FirstAscentYear,
Pk.FirstAscentCountry
FROM getdata.calculated.CalcPeaks AS Pk;

COMMENT ON TABLE getdata.calculated.CalcDimPeak IS
'მწვერვალების განზომილება დეშბორდისთვის. გასაღები: PeakId.';

In [0]:
%sql
-- 2. CalcDimYear — წლები, ათწლეულები, ეპოქები
-- მხოლოდ ის წლები, როცა ექსპედიცია ნამდვილად შედგა.

CREATE OR REPLACE TABLE getdata.calculated.CalcDimYear AS
SELECT DISTINCT
Ex.ExpeditionYear,
Ex.Decade,
CONCAT(CAST(Ex.Decade AS STRING), '-იანები') AS DecadeLabel,
Ex.Era
FROM getdata.calculated.CalcExpeditions AS Ex;

COMMENT ON TABLE getdata.calculated.CalcDimYear IS
'წლების განზომილება ათწლეულითა და ეპოქით. გასაღები: ExpeditionYear.';

In [0]:
%sql

CREATE OR REPLACE TABLE getdata.calculated.CalcDimDecade AS
SELECT DISTINCT
Dy.Decade,
Dy.DecadeLabel,
Dy.Era
FROM getdata.calculated.CalcDimYear AS Dy;

COMMENT ON TABLE getdata.calculated.CalcDimDecade IS
'ათწლეულების განზომილება. გასაღები: Decade (= CalcMembers.Decade).';

In [0]:
%sql

CREATE OR REPLACE TABLE getdata.calculated.CalcDimNation AS
WITH NationVolume AS (
SELECT
Mb.Citizenship,
COUNT(*) AS ClimberCount
FROM getdata.calculated.CalcMembers AS Mb
WHERE Mb.Citizenship IS NOT NULL
GROUP BY Mb.Citizenship
)
SELECT
Nt.Citizenship AS Nation,
Nt.ClimberCount,
CASE
WHEN Nt.ClimberCount >= 1000 THEN 'დიდი (1000+)'
WHEN Nt.ClimberCount >= 100 THEN 'საშუალო (100–999)'
ELSE 'მცირე (<100)'
END AS NationSizeBand,
Nt.Citizenship = 'Georgia' AS IsGeorgia
FROM NationVolume AS Nt;

COMMENT ON TABLE getdata.calculated.CalcDimNation IS
'მოქალაქეობების განზომილება. გასაღები: Nation (= CalcMembers.Citizenship). IsGeorgia გამოყოფს ქართველ მთამსვლელებს.';

In [0]:
%sql
-- კავშირების შემოწმება:
-- ყოველ ფაქტს უნდა ჰქონდეს შესაბამისი განზომილება.
-- სამივე რიცხვი 0 უნდა იყოს.

SELECT
SUM(CASE WHEN Dp.PeakId IS NULL THEN 1 ELSE 0 END) AS MembersWithoutPeak,
SUM(CASE WHEN Dy.ExpeditionYear IS NULL THEN 1 ELSE 0 END) AS MembersWithoutYear,
SUM(
CASE
WHEN Mb.Citizenship IS NOT NULL AND Dn.Nation IS NULL
THEN 1
ELSE 0
END
) AS MembersWithoutNation,
SUM(
CASE
WHEN NOT Mb.IsHiredStaff AND Ap.Agency IS NULL
THEN 1
ELSE 0
END
) AS ClientsWithoutAgency
FROM getdata.calculated.CalcMembers AS Mb
LEFT JOIN getdata.calculated.CalcDimPeak AS Dp
ON Mb.PeakId = Dp.PeakId
LEFT JOIN getdata.calculated.CalcDimYear AS Dy
ON Mb.ExpeditionYear = Dy.ExpeditionYear
LEFT JOIN getdata.calculated.CalcDimNation AS Dn
ON Mb.Citizenship = Dn.Nation
LEFT JOIN getdata.calculated.CalcAgencyPerformance AS Ap
ON Mb.Agency = Ap.Agency;

In [0]:
%sql
SELECT
'CalcDimPeak' AS Dimension,
COUNT(*) AS RowCount,
COUNT(DISTINCT PeakId) AS KeyCount
FROM getdata.calculated.CalcDimPeak

UNION ALL

SELECT
'CalcDimYear',
COUNT(*),
COUNT(DISTINCT ExpeditionYear)
FROM getdata.calculated.CalcDimYear

UNION ALL

SELECT
'CalcDimNation',
COUNT(*),
COUNT(DISTINCT Nation)
FROM getdata.calculated.CalcDimNation

UNION ALL

SELECT
'CalcAgencyPerformance',
COUNT(*),
COUNT(DISTINCT Agency)
FROM getdata.calculated.CalcAgencyPerformance;